# 06 - Popolazione di 10 adulti virtuali e robustezza al rumore del sensore

**Obiettivo.** Estendere il confronto, come richiesto dalla co-relatrice, a ulteriori pazienti virtuali e semi casuali:
1. tutti i 10 adulti di simglucose (`adult#001`-`adult#010`), pasti non annunciati da 60 e 90 g e verifica a digiuno,
   con i moduli attivati separatamente e insieme;
2. 10 semi del rumore del sensore CGM su `adult#001`, `adult#002`, `adult#003` (60 g).

I controllori **non** vengono ritarati sul singolo paziente: cambiano solo le grandezze legate al basale
(limiti dell'integrale e tetto IOB). Il messaggio da verificare è il **compromesso** tra riduzione dell'ipoglicemia
(insulin stacking) e peggioramento del controllo postprandiale, paziente per paziente.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
import run_campaign as RC          # stesse funzioni, stessa cache e simulazioni in parallelo della campagna
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_columns", 40)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = RC.COL
print("Versioni delle librerie usate:"); display(U.library_versions())

Versioni delle librerie usate:


,Pacchetto,Versione
0,Python,3.13.16
1,simglucose,0.2.11
2,numpy,2.4.4
3,scipy,1.17.1
4,pandas,3.0.2
5,matplotlib,3.10.8


## 1. Configurazione
### 1.1 Pazienti (parametri di simglucose)

In [2]:
rows = []
for p in RC.POPOLAZIONE:
    ic = U.initial_conditions(p, SEED)
    rows.append({"Paziente": p, "BW [kg]": round(ic["BW"], 1), "Gb [mg/dL]": round(ic["Gb"], 1),
                 "Basale u_b [U/h]": round(ic["ub"] * 60, 3), "Tetto IOB [U]": round(A.IOB_ALPHA * ic["ub"] * 60, 2)})
display(pd.DataFrame(rows))
display(U.scenario_table("adult#001", "0, 60 o 90", SEED).iloc[6:])

,Paziente,BW [kg],Gb [mg/dL],Basale u_b [U/h],Tetto IOB [U]
0,adult#001,102.3,138.6,1.267,0.95
1,adult#002,111.1,136.4,1.370,1.03
2,adult#003,81.6,147.1,1.425,1.07
3,adult#004,63.0,150.7,0.888,0.67
4,adult#005,94.1,142.7,1.180,0.88
5,adult#006,66.1,135.6,1.725,1.29
6,adult#007,91.2,135.3,1.371,1.03
7,adult#008,102.8,143.2,1.141,0.86
8,adult#009,74.6,145.1,1.134,0.85
9,adult#010,73.9,152.8,1.019,0.76


,Voce,Valore,Unita'
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00 del giorno precedente (24 h prima),ora del giorno
8,Assestamento escluso dalle metriche,giorno precedente + 00:00-08:00 (32 h),-
9,Finestra di analisi,08:00-24:00,ora del giorno
10,Passo di campionamento dt,3.0,min
11,Passi totali / nella finestra,960 / 320,-
12,Glicemia plasmatica iniziale (inizio simulazione),138.6,mg/dL
13,Misura CGM iniziale (inizio simulazione),152.0,mg/dL
14,"Stati iniziali stimatore IOB (S1, S2)","0.0, 0.0",U
15,Orario del pasto,12:00,ora del giorno


### 1.2 Parametri dei controllori (con unità di misura)

In [3]:
ub = U.initial_conditions("adult#001", SEED)["ub"]
tabs = {n: U.controller_table(A.CFG[n], ub).set_index("Parametro") for n in RC.MODULI}
cmp_ = pd.DataFrame({n: tabs[n]["Valore"].astype(str) for n in RC.MODULI}); cmp_["Unita'"] = tabs["PID"]["Unita'"]
display(cmp_)

,PID,PID cons.,PID+IOB,PID+PLGS,PID+IOB+PLGS,Unita'
Parametro,,,,,,
target,120.0,120.0,120.0,120.0,120.0,mg/dL
kp,0.0001,0.0001,0.0001,0.0001,0.0001,U/min per mg/dL
ki,3e-07,3e-07,3e-07,3e-07,3e-07,U/min per (mg/dL*min)
kd,0.002,0.002,0.002,0.002,0.002,U/min per (mg/dL/min)
n_slope,5,5,5,5,5,campioni
aw_frac,1.0,0.5,1.0,1.0,1.0,adimensionale
aw_frac_low,0.25,0.25,0.25,0.25,0.25,adimensionale
freeze,True,True,True,True,True,-
use_iob,False,False,True,False,True,-


### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `Vincolo` | - | vincolo attivo nel passo: nessuno, `iob`, `plgs`, `sat_alta`, `sat_bassa` |

## 2. Popolazione: simulazioni

In [4]:
CARICHI = [0, 60, 90]
keys = [RC.key(p, c, n) for p in RC.POPOLAZIONE for c in CARICHI for n in RC.MODULI]
RC.run_all(keys)
Pop = pd.DataFrame([RC.row(k) for k in keys]).round(1)
print(len(keys), "simulazioni")

150 simulazioni


### 2.1 Sintesi sulla popolazione (media ± deviazione standard sui 10 pazienti)

In [5]:
cols = [RC.COLONNE[c] for c in ["TIR", "TAR", "TBR", "TBR_tardivo", "picco", "IAE_range", "t_rientro", "n_commut", "TV_u"]]
for c in CARICHI:
    print(f"Pasto da {c} g:" if c else "A digiuno (0 g):")
    display(RC.media_std(Pop[Pop["CHO [g]"] == c], ["Controllore"], cols))

A digiuno (0 g):


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,100.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,120.1 ± 1.8,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,8.2 ± 0.0,0
1,PID cons.,100.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,120.1 ± 1.8,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,8.2 ± 0.0,0
2,PID+IOB,100.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,120.1 ± 1.8,0.0 ± 0.0,0.0 ± 0.0,1.7 ± 5.4,9.1 ± 2.7,0
3,PID+PLGS,100.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,122.5 ± 4.7,0.0 ± 0.0,0.0 ± 0.0,4.0 ± 0.0,13.0 ± 0.9,0
4,PID+IOB+PLGS,100.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,122.5 ± 4.7,0.0 ± 0.0,0.0 ± 0.0,7.2 ± 10.1,14.7 ± 5.5,0


Pasto da 60 g:


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,75.7 ± 11.6,22.1 ± 9.8,2.2 ± 5.1,4.0 ± 9.0,215.9 ± 27.4,95.6 ± 77.1,4.8 ± 2.2,0.0 ± 0.0,8.6 ± 0.2,1
1,PID cons.,76.4 ± 11.8,22.2 ± 9.8,1.5 ± 4.7,2.7 ± 8.4,215.9 ± 27.4,95.8 ± 77.4,4.3 ± 1.9,0.0 ± 0.0,8.6 ± 0.2,1
2,PID+IOB,75.4 ± 11.1,24.5 ± 11.1,0.1 ± 0.3,0.2 ± 0.5,217.9 ± 27.9,111.2 ± 88.0,4.7 ± 2.0,80.9 ± 69.2,81.0 ± 67.0,1
3,PID+PLGS,78.6 ± 10.0,21.4 ± 10.0,0.0 ± 0.0,0.0 ± 0.0,213.3 ± 27.0,86.5 ± 72.6,4.3 ± 1.8,6.8 ± 2.7,19.7 ± 5.0,0
4,PID+IOB+PLGS,76.1 ± 11.4,23.9 ± 11.4,0.0 ± 0.0,0.0 ± 0.0,215.4 ± 27.7,104.2 ± 87.8,4.7 ± 2.0,88.0 ± 67.3,90.8 ± 64.8,0


Pasto da 90 g:


,Controllore,TIR [%],TAR [%],TBR [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Variazione totale infusione [U/h],Non rientra [n]
0,PID,62.4 ± 11.8,28.5 ± 7.0,9.0 ± 12.1,16.1 ± 21.5,265.0 ± 41.2,277.7 ± 153.1,7.3 ± 2.9,0.0 ± 0.0,9.0 ± 0.3,2
1,PID cons.,64.9 ± 10.7,28.8 ± 7.2,6.3 ± 10.6,11.3 ± 18.8,265.0 ± 41.3,274.6 ± 151.5,6.6 ± 2.1,0.0 ± 0.0,9.0 ± 0.4,1
2,PID+IOB,64.9 ± 8.3,33.6 ± 8.9,1.5 ± 4.7,2.7 ± 8.4,270.2 ± 42.3,326.9 ± 185.7,6.7 ± 1.8,112.1 ± 50.9,128.8 ± 62.2,0
3,PID+PLGS,68.5 ± 7.9,28.2 ± 7.2,3.4 ± 5.5,6.0 ± 9.8,262.1 ± 40.6,255.2 ± 143.7,6.5 ± 2.0,8.0 ± 2.3,22.6 ± 2.9,0
4,PID+IOB+PLGS,66.7 ± 9.2,33.3 ± 9.2,0.0 ± 0.0,0.0 ± 0.0,267.7 ± 42.0,316.4 ± 186.7,6.1 ± 1.4,116.9 ± 47.3,136.4 ± 59.4,0


### 2.2 Distribuzione delle metriche sui pazienti

In [6]:
MET = ["TIR [%]", "TBR [%]", "Picco [mg/dL]", "IAE fuori range [mg/dL*h]", "Commutazioni vincolo [-]"]
fig, ax = plt.subplots(2, len(MET), figsize=(18, 7))
for i, c in enumerate([60, 90]):
    d = Pop[Pop["CHO [g]"] == c]
    for j, colm in enumerate(MET):
        dati = [d[d.Controllore == n][colm].values for n in RC.MODULI]
        bp = ax[i, j].boxplot(dati, patch_artist=True, widths=0.6)
        for patch, n in zip(bp["boxes"], RC.MODULI): patch.set_facecolor(COL[n]); patch.set_alpha(0.5)
        for k, y in enumerate(dati, 1): ax[i, j].scatter(np.full(len(y), k), y, s=10, color="k", zorder=3)
        ax[i, j].set_xticks(range(1, len(RC.MODULI) + 1)); ax[i, j].set_xticklabels(RC.MODULI, rotation=30, ha="right", fontsize=8)
        ax[i, j].set_title(f"{colm}, {c} g", fontsize=9); ax[i, j].grid(ls=":", alpha=0.5)
plt.tight_layout(); plt.show()

### 2.3 Compromesso paziente per paziente: `PID+IOB+PLGS` meno `PID`

In [7]:
D = []
for c in [60, 90]:
    d = Pop[Pop["CHO [g]"] == c].set_index(["Paziente", "Controllore"])
    for p in RC.POPOLAZIONE:
        a, b = d.loc[(p, "PID")], d.loc[(p, "PID+IOB+PLGS")]
        D.append({"Paziente": p, "CHO [g]": c, **{f"Δ {m}": round(b[m] - a[m], 1) for m in
                  ["TIR [%]", "TAR [%]", "TBR [%]", "TBR tardivo [%]", "Picco [mg/dL]", "IAE fuori range [mg/dL*h]"]}})
D = pd.DataFrame(D); display(D)
fig, ax = plt.subplots(figsize=(7, 5.5))
for c, colore in zip([60, 90], ["tab:blue", "tab:red"]):
    d = D[D["CHO [g]"] == c]
    ax.scatter(d["Δ TAR [%]"], d["Δ TBR [%]"], color=colore, label=f"{c} g")
    for _, r in d.iterrows(): ax.annotate(r.Paziente[-2:], (r["Δ TAR [%]"], r["Δ TBR [%]"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8); ax.grid(ls=":", alpha=0.5); ax.legend()
ax.set_xlabel("Δ TAR [punti %] (proposto − PID)"); ax.set_ylabel("Δ TBR [punti %] (proposto − PID)")
ax.set_title("In basso a destra: meno ipoglicemia al prezzo di più iperglicemia"); plt.tight_layout(); plt.show()

,Paziente,CHO [g],Δ TIR [%],Δ TAR [%],Δ TBR [%],Δ TBR tardivo [%],Δ Picco [mg/dL],Δ IAE fuori range [mg/dL*h]
0,adult#001,60,-0.3,0.3,0.0,0.0,-0.5,-0.9
1,adult#002,60,0.0,0.0,0.0,0.0,-0.8,-1.2
2,adult#003,60,-1.0,1.0,0.0,0.0,2.5,1.7
3,adult#004,60,0.9,6.6,-7.5,-13.3,-4.3,25.1
4,adult#005,60,-3.1,3.1,0.0,0.0,4.2,20.2
5,adult#006,60,14.3,0.6,-15.0,-26.7,-4.1,-24.4
6,adult#007,60,3.8,-3.8,0.0,0.0,-11.5,-25.6
7,adult#008,60,0.0,0.0,0.0,0.0,-0.8,0.0
8,adult#009,60,-4.1,4.1,0.0,0.0,6.9,36.4
9,adult#010,60,-6.3,6.3,0.0,0.0,3.5,54.4


## 3. Robustezza al rumore del sensore: 10 semi (60 g)

In [8]:
sk = [RC.key(p, 60, n, seed=s) for p in RC.PAZIENTI for n in ["PID", "PID cons.", "PID+IOB+PLGS"] for s in RC.SEMI]
RC.run_all(sk)
Sd = pd.DataFrame([{**RC.row(k), "Seme": k[3]} for k in sk])
display(RC.media_std(Sd, ["Paziente", "Controllore"],
        [RC.COLONNE[c] for c in ["TIR", "TAR", "TBR", "TBR<54", "TBR_tardivo", "picco", "IAE_range", "t_rientro", "n_commut"]]))

,Paziente,Controllore,TIR [%],TAR [%],TBR [%],TBR<54 [%],TBR tardivo [%],Picco [mg/dL],IAE fuori range [mg/dL*h],Rientro in range [h],Commutazioni vincolo [-],Non rientra [n]
0,adult#001,PID,74.9 ± 2.5,25.1 ± 2.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,199.0 ± 3.3,33.3 ± 11.0,5.1 ± 0.1,0.0 ± 0.0,0
1,adult#001,PID cons.,74.9 ± 2.5,25.1 ± 2.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,199.0 ± 3.3,33.3 ± 11.0,5.1 ± 0.1,0.0 ± 0.0,0
2,adult#001,PID+IOB+PLGS,75.9 ± 6.5,24.1 ± 6.5,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,199.7 ± 5.5,37.9 ± 19.7,5.2 ± 0.2,68.0 ± 11.7,0
3,adult#002,PID,82.6 ± 1.2,17.4 ± 1.2,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,202.1 ± 3.2,38.2 ± 8.4,4.2 ± 0.1,0.0 ± 0.0,0
4,adult#002,PID cons.,82.6 ± 1.2,17.4 ± 1.2,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,202.1 ± 3.2,38.2 ± 8.4,4.2 ± 0.1,0.0 ± 0.0,0
5,adult#002,PID+IOB+PLGS,82.0 ± 1.9,18.0 ± 1.9,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,203.0 ± 5.3,41.9 ± 14.7,4.3 ± 0.2,40.6 ± 6.8,0
6,adult#003,PID,73.5 ± 1.1,26.5 ± 1.1,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,207.5 ± 5.4,82.3 ± 20.6,5.1 ± 0.1,0.0 ± 0.0,0
7,adult#003,PID cons.,73.5 ± 1.1,26.5 ± 1.1,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,207.5 ± 5.4,82.3 ± 20.6,5.1 ± 0.1,0.0 ± 0.0,0
8,adult#003,PID+IOB+PLGS,71.8 ± 1.6,28.2 ± 1.6,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0,212.2 ± 8.4,98.8 ± 39.3,5.3 ± 0.1,69.8 ± 9.9,0


## 4. Lettura dei risultati (calcolata dalle tabelle)

In [9]:
for c in [60, 90]:
    d = D[D["CHO [g]"] == c]
    meno_ipo = d[d["Δ TBR [%]"] < 0]; piu_iper = d[d["Δ TAR [%]"] > 0]
    entrambi = d[(d["Δ TBR [%]"] < 0) & (d["Δ TAR [%]"] > 0)]
    print(f"{c} g: il proposto riduce il TBR in {len(meno_ipo)}/10 pazienti, aumenta il TAR in {len(piu_iper)}/10, "
          f"entrambe le cose (compromesso) in {len(entrambi)}/10; dTIR medio {d['Δ TIR [%]'].mean():+.1f} punti, "
          f"dTBR medio {d['Δ TBR [%]'].mean():+.1f}, dTAR medio {d['Δ TAR [%]'].mean():+.1f}")

60 g: il proposto riduce il TBR in 2/10 pazienti, aumenta il TAR in 7/10, entrambe le cose (compromesso) in 2/10; dTIR medio +0.4 punti, dTBR medio -2.2, dTAR medio +1.8
90 g: il proposto riduce il TBR in 5/10 pazienti, aumenta il TAR in 10/10, entrambe le cose (compromesso) in 5/10; dTIR medio +4.3 punti, dTBR medio -9.0, dTAR medio +4.7


**Limiti.** Simulazioni *in silico* con un solo pasto al giorno e guadagni non ritarati sui singoli pazienti:
i risultati mostrano il comportamento di parametri fissi su una popolazione eterogenea, non il miglior risultato
ottenibile per ciascun soggetto. Il controllore proposto **non** va presentato come uniformemente migliore: l'analisi
serve a mostrare dove e quanto il compromesso tra sicurezza e controllo dell'iperglicemia si sposta.